In [3]:
import polars as pl
import os
from pathlib import Path

# Polars 출력 설정 - 모든 행과 컬럼 표시
pl.Config.set_tbl_rows(999)
pl.Config.set_tbl_cols(999)
pl.Config.set_tbl_width_chars(120)

# CSV 파일의 절대경로
notebook_dir = Path('.').resolve()
csv_path = notebook_dir.parent / 'output' / 'all_trials_timeseries.csv'

print(f"Current dir: {notebook_dir}")
print(f"CSV Path: {csv_path}")
print(f"Exists: {csv_path.exists()}")

# CSV 읽기
df = pl.read_csv(csv_path)

print("\n" + "="*80)
print("STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)")
print("="*80)

# Trial별 step_TF 결정 (각 trial은 하나의 step_TF 값을 가짐)
trial_level = df.select(['subject', 'velocity', 'trial', 'step_TF']).unique().sort(
    ['subject', 'velocity', 'trial']
)

print("\n[1] Trial별 Step/Nonstep 분포")
print("-" * 50)
trial_step_dist = trial_level.group_by('step_TF').agg(
    pl.len().alias('trial_count')
).with_columns(
    ratio_percent = (pl.col('trial_count') / pl.col('trial_count').sum() * 100).round(2)
).sort('step_TF', descending=True)
print(trial_step_dist)

print("\n[2] 인당 Trial 기술통계 (mean, sd, min, max)")
print("-" * 50)

# step / nonstep
desc_per_subject = (
    trial_level
    .group_by(['subject', 'step_TF'])
    .agg(pl.len().alias('trial_count'))
    .group_by('step_TF')
    .agg(
        pl.col('trial_count').mean().round(2).alias('mean'),
        pl.col('trial_count').std().round(2).alias('sd'),
        pl.col('trial_count').min().alias('min'),
        pl.col('trial_count').max().alias('max'),
    )
    .sort('step_TF', descending=True)
)

# All_trials (피험자별 전체 trial 수)
all_trials_desc = (
    trial_level
    .group_by('subject')
    .agg(pl.len().alias('trial_count'))
    .select(
        pl.lit('All_trials').alias('step_TF'),
        pl.col('trial_count').mean().round(2).alias('mean'),
        pl.col('trial_count').std().round(2).alias('sd'),
        pl.col('trial_count').min().alias('min'),
        pl.col('trial_count').max().alias('max'),
    )
)

desc_combined = pl.concat([desc_per_subject, all_trials_desc])
print(desc_combined)


Current dir: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\analysis
CSV Path: C:\Users\Alice\OneDrive - 청주대학교\근전도 분석 코드\replace_V3D\output\all_trials_timeseries.csv
Exists: True

STEP vs NONSTEP 분석 (Subject-Velocity-Trial 단위)

[1] Trial별 Step/Nonstep 분포
--------------------------------------------------
shape: (2, 3)
┌─────────┬─────────────┬───────────────┐
│ step_TF ┆ trial_count ┆ ratio_percent │
│ ---     ┆ ---         ┆ ---           │
│ str     ┆ u32         ┆ f64           │
╞═════════╪═════════════╪═══════════════╡
│ step    ┆ 53          ┆ 42.4          │
│ nonstep ┆ 72          ┆ 57.6          │
└─────────┴─────────────┴───────────────┘

[2] 인당 Trial 기술통계 (mean, sd, min, max)
--------------------------------------------------
shape: (3, 5)
┌────────────┬──────┬──────┬─────┬─────┐
│ step_TF    ┆ mean ┆ sd   ┆ min ┆ max │
│ ---        ┆ ---  ┆ ---  ┆ --- ┆ --- │
│ str        ┆ f64  ┆ f64  ┆ u32 ┆ u32 │
╞════════════╪══════╪══════╪═════╪═════╡
│ step       ┆ 2.52 ┆ 1.33 

# 피험자별 Step/Nonstep 데이터

In [4]:
print("\n[2] 피험자별 Step/Nonstep Trial 개수")
print("-" * 50)
subject_step_count = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count'),
    pl.col('velocity').n_unique().alias('velocity_type_count')
).sort(['subject', 'step_TF'])
print(subject_step_count)

print("\n[3] 피험자별 총 Trial 개수")
print("-" * 50)
subject_total = trial_level.group_by('subject').agg(
    pl.len().alias('total_trial_count'),
    pl.col('velocity').n_unique().alias('velocity_count')
).sort('subject')
print(subject_total)

print("\n[4] 상세 Trial 목록 (Subject-Velocity-Trial-Step)")
print("-" * 50)
detailed = trial_level.sort(['subject', 'velocity', 'trial'])
print(detailed)

print("\n[5] 피험자별 Step/Nonstep Trial 개수 Summary")
print("-" * 50)
subject_summary = trial_level.group_by(['subject', 'step_TF']).agg(
    pl.len().alias('trial_count')
).sort(['subject', 'step_TF'])
print(subject_summary)


[2] 피험자별 Step/Nonstep Trial 개수
--------------------------------------------------
shape: (45, 4)
┌─────────┬─────────┬─────────────┬─────────────────────┐
│ subject ┆ step_TF ┆ trial_count ┆ velocity_type_count │
│ ---     ┆ ---     ┆ ---         ┆ ---                 │
│ str     ┆ str     ┆ u32         ┆ u32                 │
╞═════════╪═════════╪═════════════╪═════════════════════╡
│ 가윤호  ┆ nonstep ┆ 4           ┆ 1                   │
│ 가윤호  ┆ step    ┆ 2           ┆ 1                   │
│ 강비은  ┆ nonstep ┆ 5           ┆ 1                   │
│ 강비은  ┆ step    ┆ 3           ┆ 1                   │
│ 권유영  ┆ nonstep ┆ 1           ┆ 1                   │
│ 김민정  ┆ nonstep ┆ 2           ┆ 1                   │
│ 김민정  ┆ step    ┆ 4           ┆ 1                   │
│ 김서하  ┆ nonstep ┆ 6           ┆ 1                   │
│ 김서하  ┆ step    ┆ 3           ┆ 1                   │
│ 김우연  ┆ nonstep ┆ 1           ┆ 1                   │
│ 김우연  ┆ step    ┆ 1           ┆ 1                   │
│ 김유민  

# Step onset 평균 
mean / sd / min / max 

In [ ]:

# Step onset 기술통계 [1]: 전체 (초, platform onset 기준)
step_onset_df = (
    df
    .filter(pl.col('step_TF') == 'step')
    .filter(pl.col('Is_step_onset_frame'))
    .select(['subject', 'velocity', 'trial', 'step_onset_local', 'platform_onset_local'])
    .with_columns(
        pl.col('step_onset_local').cast(pl.Int64)
    )
    .with_columns(
        step_onset_s=(pl.col('step_onset_local') - pl.col('platform_onset_local')) / 1000.0
    )
    .sort(['subject', 'velocity', 'trial'])
)

print(f"Step trials 수: {step_onset_df.height}")
print()

print("[1] Step onset 전체 기술통계 (초, platform onset 기준)")
print("-" * 50)
overall = step_onset_df.select(
    pl.col('step_onset_s').mean().alias('mean'),
    pl.col('step_onset_s').std().alias('sd'),
    pl.col('step_onset_s').min().alias('min'),
    pl.col('step_onset_s').max().alias('max'),
).with_columns(pl.all().round(4))
print(overall)
